# COde Dataset Audit — Research Design & Patient-Level Split

## Thesis milestone

**Roadmap position:** Research Design → **COde Dataset Audit** → Patient-Level Split → Data Pipeline → ...

This notebook documents the dataset audit and leakage-risk investigation performed before model development. The goal was to determine whether the COde dataset can be safely split at the image/visit level or whether the split must be performed at the patient level.

**Important:** This notebook is an audit and documentation artifact. It does not train a predictive model and does not claim that every repeated image represents an invalid record. The objective is to establish a defensible data-splitting policy and document the evidence behind it.


## 1. Research Question

The central question of this milestone was:

> **Can samples from the same patient be allowed to appear across train, validation, and test sets, or must all samples belonging to one patient remain in a single split?**

This matters because the dataset contains longitudinal visits and repeated/reused images across visits. A random image-level or visit-level split can therefore allow information from the same patient to appear in multiple splits, producing optimistic evaluation and potentially invalid generalization estimates.


## 2. Audit Workflow

The audit was performed in several stages:

1. Inspect the raw COde dataset structure and metadata.
2. Normalize and audit patient identifiers.
3. Investigate repeated images within and across visits.
4. Characterize cross-visit image reuse within the same patient.
5. Compare clinical/treatment context of reused images.
6. Investigate tooth-level overlap and treatment-episode relationships.
7. Review a stratified sample of ambiguous episode relationships.
8. Use the evidence to determine the safest split policy.

The analysis was intentionally conservative: it was designed to answer the split question, not to prove that every image reuse event is clinically invalid.


## 3. Dataset-Level Findings

The audit operated on the COde dataset containing approximately **8,775 visit-level records** and **35 columns**.

The audit pipeline identified substantial patient-level longitudinal structure and cross-visit image reuse.

Key outputs included:

- patient/image trace audit results
- cross-visit image reuse characterization
- treatment and clinical context characterization
- tooth overlap analysis
- treatment episode relation analysis
- stratified manual-review sample

The important point for research design is that the dataset cannot safely be treated as a collection of independent images or independent visits.


## 4. Cross-Visit Image Reuse

The patient treatment characterization stage loaded **1,589 reused visit-pair rows** involving **826 patients**.

Observed relationships included:

- Same treatment pairs: **2**
- Different treatment pairs: **47**
- Exact tooth-overlap pairs: **43**
- Partial tooth-overlap pairs: **190**
- No tooth-overlap pairs: **226**
- Likely same treatment episode pairs: **111**
- Possible same treatment episode pairs: **773**
- Likely different treatment episode pairs: **1**
- Same-day reuse pairs: **0**
- Long-term reuse pairs: **12**

These counts should be interpreted as characterization outputs rather than definitive clinical labels. In particular, the large number of `possible_same_episode` cases and missing tooth information show that the exact clinical meaning of every reuse event cannot be established automatically from the available metadata.


## 5. Tooth-Level and Episode-Level Findings

The detailed characterization contained 1,589 reused visit-pair rows.

### Tooth overlap

| Relationship | Count |
|---|---:|
| One side missing tooth information | 863 |
| Both sides missing tooth information | 267 |
| No tooth overlap | 226 |
| Partial overlap | 190 |
| Exact overlap | 43 |

### Episode relation

| Relationship | Count |
|---|---:|
| Possible same episode | 773 |
| Insufficient evidence | 477 |
| Possible different episode | 227 |
| Likely same episode | 111 |
| Likely different episode | 1 |

The main methodological observation is that **the available metadata are insufficient to confidently classify all reused images into clinically independent or dependent episodes**. This is especially clear from the large `insufficient_evidence` group and the high proportion of missing tooth information.


## 6. Stratified Review of Episode Relationships

A stratified review sample of **81 reused visit pairs** was generated to inspect the heuristic episode classifications.

Sampling was performed across:

- 30 `possible_same_episode`
- 20 `insufficient_evidence`
- 20 `possible_different_episode`
- 10 `likely_same_episode`
- 1 `likely_different_episode`

The review showed that many pairs had missing tooth information, while several clearly shared teeth or had overlapping clinical/treatment context. The evidence therefore supports the existence of meaningful patient-level longitudinal dependence, but it does **not** provide a reliable rule for declaring reused images independent enough to permit patient splitting across datasets.


## 7. Main Research Design Decision

### Decision: Patient-level splitting

All samples belonging to the same patient should remain in exactly one of the train, validation, or test partitions.

Conceptually:

```text
Patient A ───────────────> Train
Patient B ───────────────> Validation
Patient C ───────────────> Test
```

and **never**:

```text
Patient A ──> Train
Patient A ──> Test
```

The split unit is therefore the **patient**, not the image and not the visit.


## 8. Why Patient-Level Split Is the Defensible Choice

The evidence does not require us to prove that every reused image is clinically identical or that every repeated visit is dependent. Instead, the key issue is whether we can **reliably guarantee independence across splits**.

The audit indicates that we cannot make that guarantee because:

1. Cross-visit image reuse exists within the same patient.
2. A substantial number of reused-image pairs have overlapping or potentially related treatment context.
3. Tooth information is frequently missing, preventing reliable tooth-level independence checks.
4. Episode classification is uncertain for a large fraction of cases.
5. The dataset is longitudinal, so multiple visits from the same patient are not naturally independent observations.

Therefore, the conservative and methodologically defensible policy is to keep each patient entirely within one split.


## 9. What We Do NOT Claim

This audit does **not** claim that:

- every reused image is a duplicate in the problematic sense;
- every repeated visit represents the same treatment episode;
- all cross-visit reuse is clinically invalid;
- patient-level splitting completely eliminates every possible form of dataset leakage.

The conclusion is narrower and stronger:

> **Given the observed longitudinal structure, cross-visit image reuse, incomplete clinical metadata, and uncertainty in episode-level independence, patient-level splitting is the safest and most defensible evaluation protocol for this study.**


## 10. Thesis-Ready Evidence to Preserve

The following artifacts should be retained for the thesis and paper:

### Core evidence
- Dataset size and column structure from the initial audit.
- Patient ID normalization and uniqueness checks.
- Cross-visit image reuse statistics.
- Number of affected patients.
- Treatment-context characterization.
- Tooth-overlap characterization.
- Episode-relation characterization.
- Stratified review sample and its methodology.

### Core outputs
- `duplicated_visit_treatment.csv`
- `treatment_pair_summary.csv`
- `episode_pair_summary.csv`
- `patient_treatment_longitudinal.csv`
- `treatment_keyword_frequency.csv`
- `audit_summary.json`
- `episode_relation_review_sample.csv`

These files provide an auditable trail from raw dataset structure to the final split decision.


## 11. Recommended Split Policy for the Next Milestone

The next implementation stage should create a deterministic **patient-level split manifest**.

Recommended structure:

| patient_id | split |
|---|---|
| 0001 | train |
| 0002 | train |
| 0003 | validation |
| 0004 | test |

The manifest should be generated once using a fixed random seed and then reused by every downstream experiment.

All later experiments—including supervised baselines, SSL, multimodal fusion, missing-modality experiments, ablations, and final evaluation—must use the same patient-level split policy unless a specific experiment has a documented reason to deviate.


## 12. Final Milestone Conclusion

### Status: COMPLETE

The purpose of the COde Dataset Audit milestone was to establish a defensible data-splitting strategy before model development.

The audit demonstrated that the dataset contains substantial patient-level longitudinal structure and cross-visit image reuse, while the available metadata are not sufficient to reliably establish independence between all repeated visits or reused images.

### Final decision

**Use patient-level splitting.**

Further investigation into the exact clinical validity of every reuse event is not necessary for deciding the split strategy. The detailed treatment and episode analysis served its purpose by showing that a reliable independence rule at the image/visit level cannot be established from the available metadata.

The project can therefore move forward to the next milestone:

**Patient-Level Split → Data Pipeline**


In [ ]:
{
echo "===== 13-LABEL RADIOGRAPH ONLY ====="
for f in \
results/baseline/radiograph_only/evaluation.json \
results/baseline/radiograph_only/history.json \
results/baseline/radiograph_only/thresholds.json
do
  echo
  echo "===== $f ====="
  cat "$f"
done

echo
echo "===== SIX-LABEL DATASET ====="
for f in \
results/six_label_patient_level_dataset/dataset_summary.json \
results/six_label_patient_level_dataset/label_split_distribution.csv
do
  echo
  echo "===== $f ====="
  cat "$f"
done

echo
echo "===== RADIOGRAPH ONLY 6-LABEL ====="
for f in \
results/baseline/radiograph_only_6label/test_evaluation.json \
results/baseline/radiograph_only_6label/validation_evaluation.json \
results/baseline/radiograph_only_6label/thresholds.json \
results/baseline/radiograph_only_6label/history.json
do
  echo
  echo "===== $f ====="
  cat "$f"
done

echo
echo "===== PHOTOGRAPH ONLY 6-LABEL ====="
for f in \
results/baseline/photograph_only_6label/test_evaluation.json \
results/baseline/photograph_only_6label/validation_evaluation.json \
results/baseline/photograph_only_6label/thresholds.json \
results/baseline/photograph_only_6label/history.json
do
  echo
  echo "===== $f ====="
  cat "$f"
done

echo
echo "===== TEXT ONLY 6-LABEL ====="
for f in \
results/baseline/text_only_6label/test_evaluation.json \
results/baseline/text_only_6label/test_metrics.json \
results/baseline/text_only_6label/test_metrics_optimized.json \
results/baseline/text_only_6label/thresholds.json \
results/baseline/text_only_6label/history.json
do
  echo
  echo "===== $f ====="
  cat "$f"
done

echo
echo "===== FULL MULTIMODAL 6-LABEL ====="
for f in \
results/baseline/full_multimodal_6label/test_metrics.json \
results/baseline/full_multimodal_6label/test_metrics_thresholded.json \
results/baseline/full_multimodal_6label/per_label_metrics.json \
results/baseline/full_multimodal_6label/per_label_metrics_thresholded.json \
results/baseline/full_multimodal_6label/thresholds.json \
results/baseline/full_multimodal_6label/history.json
do
  echo
  echo "===== $f ====="
  cat "$f"
done

echo
echo "===== DATASET AUDIT ====="
for f in \
results/multimodal_dataset_audit/audit_summary.json \
results/multimodal_dataset_audit/modality_availability.csv \
results/multimodal_dataset_audit/visit_modality_summary.csv \
results/multimodal_dataset_audit/patient_modality_summary.csv \
results/patient_cross_visit_characterization/audit_summary.json \
results/patient_image_duplication_audit/audit_summary.json
do
  echo
  echo "===== $f ====="
  cat "$f"
done

echo
echo "===== SPLIT + LEAKAGE ====="
for f in \
results/patient_level_split/split_summary.json \
results/patient_level_split_leakage/leakage_summary.json \
results/patient_level_split_leakage/assignment_validation.json
do
  echo
  echo "===== $f ====="
  cat "$f"
done

} > presentation_results_v2.txt